# Phase 18D Notebook 00: Discover Kaggle Inputs\n\nThis notebook discovers the `/kaggle/input` directory by observed contents, identifies candidate metadata and source files, emits source/privacy evidence, and performs no training.

In [ ]:
authorization_flags = {"authorized": False, "real_execution_authorized": False, "publication_authorized": False, "phase_19_forbidden": True}
from __future__ import annotations
import csv, hashlib, hmac, json, os, platform, shutil, sys
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
INPUT = Path('/kaggle/input'); OUTPUT = Path('/kaggle/working/acda3d_binding')
PHASE_19_FORBIDDEN = True
SECRET = os.environ.get('ACDA3D_SUBJECT_HMAC_KEY') or os.environ.get('KAGGLE_HMAC_SECRET')
if not INPUT.is_dir(): raise RuntimeError('KAGGLE_INPUT_MISSING')
if not SECRET: raise RuntimeError('HMAC_SECRET_MISSING')
if os.environ.get('ACDA3D_REAL_EXECUTION_AUTHORIZED','false').lower() == 'true': raise RuntimeError('REAL_EXECUTION_AUTHORIZATION_MUST_REMAIN_FALSE')
if os.environ.get('ACDA3D_PUBLICATION_AUTHORIZED','false').lower() == 'true': raise RuntimeError('PUBLICATION_AUTHORIZATION_MUST_REMAIN_FALSE')
OUTPUT.mkdir(parents=True, exist_ok=True)
def sha256(path): return hashlib.sha256(path.read_bytes()).hexdigest()
def find_csv():
    files = sorted(p for p in INPUT.rglob('ad_new_2_19_2026.csv') if p.is_file())
    if not files: raise FileNotFoundError('ADNI_METADATA_MISSING')
    if len({sha256(p) for p in files}) != 1: raise RuntimeError('ADNI_METADATA_AMBIGUOUS')
    return files[0]
def choose(columns, names, label):
    for name in names:
        if name in columns: return name
    raise ValueError(f'ADNI_REQUIRED_FIELD_MISSING:{label}')
def subject_hash(value): return hmac.new(SECRET.encode(), f'ADNI:{value}'.encode(), hashlib.sha256).hexdigest()
metadata = find_csv(); raw = metadata.read_bytes(); rows = list(csv.DictReader(raw.decode('utf-8-sig').splitlines())); columns = list(rows[0]) if rows else []
subject_col = choose(columns, ('subject_id','Subject','subject','RID','PTID'), 'subject')
diagnosis_col = choose(columns, ('diagnosis','DX','Diagnosis','label','Group'), 'diagnosis')
scan_col = choose(columns, ('scan_id','image_id','ImageID','SeriesID','study_id'), 'scan')
labels = {'CN','MCI','AD'}; by_subject = defaultdict(set); counts = Counter()
for row in rows:
    subject = (row.get(subject_col) or '').strip(); diagnosis = (row.get(diagnosis_col) or '').strip().upper(); scan = (row.get(scan_col) or '').strip()
    if not subject or not scan or diagnosis not in labels: raise ValueError('ADNI_METADATA_INVALID')
    by_subject[subject].add(diagnosis); counts[diagnosis] += 1
if any(len(values) > 1 for values in by_subject.values()): raise ValueError('ADNI_CONFLICTING_SUBJECT_MAPPING')
inventory = [{'relative_path':str(p.relative_to(INPUT)).replace('\','/'),'byte_size':p.stat().st_size} for p in sorted(INPUT.rglob('*')) if p.is_file()]
provenance = {'source_url':'https://www.kaggle.com/datasets/sanjukaggling/adnidataset','observed_dataset_name':'ADNI_dataset','metadata_attestation':{'relative_path':str(metadata.relative_to(INPUT)).replace('\','/'),'sha256':sha256(metadata),'byte_size':len(raw)},'discovery_timestamp':datetime.now(timezone.utc).isoformat(),'notebook_identity':'00_kaggle_input_binding.ipynb','hmac_algorithm':'HMAC-SHA256','hmac_key_id':'acda3d-subject-id','hmac_key_version':'v1'}
manifest = {'relative_path':provenance['metadata_attestation']['relative_path'],'sha256':sha256(metadata),'byte_size':len(raw),'row_count':len(rows),'columns':columns,'required_fields':{'subject':subject_col,'diagnosis':diagnosis_col,'scan':scan_col},'label_counts':dict(counts),'canonical_subject_count':len(by_subject),'conflicting_subject_count':0,'approved_canonical_manifest':False}
environment = {'python':sys.version,'platform':platform.platform(),'pytorch':None,'cuda_available':False,'gpu':None,'cuda_version':None,'ram_bytes':None,'free_storage_bytes':shutil.disk_usage('/kaggle/working').free}
try:
    import torch
    environment.update({'pytorch':torch.__version__,'cuda_available':torch.cuda.is_available(),'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,'cuda_version':torch.version.cuda})
except ImportError: pass
for name, value in {'source_provenance.json':provenance,'metadata_manifest.json':manifest,'adni_input_inventory.json':{'files':inventory,'file_count':len(inventory)},'oasis_input_inventory.json':{'files':[],'status':'external_evidence_required'},'environment.json':environment,'binding_report.json':{'status':'KAGGLE_NOTEBOOKS_READY_FOR_EXECUTION','subject_count':len(by_subject),'raw_ids_emitted':False,'secret_emitted':False},'privacy_report.json':{'hmac_algorithm':'HMAC-SHA256','hmac_key_id':'acda3d-subject-id','hmac_key_version':'v1','raw_ids_emitted':False,'secrets_emitted':False,'key_stored_in_repository':False}}.items():
    (OUTPUT/name).write_text(json.dumps(value, indent=2), encoding='utf-8')
print(f'Bound ADNI metadata: rows={len(rows)} persons={len(by_subject)} labels={dict(counts)}')
